# Stage 1-2: Data cleaning and Teff/Tbio reconciliation

This notebook/script:
1. Loads and cleans the raw patient dataset
2. Resolves the any patient duplicate-row issue
3. Computes the corrected population Teff/Tbio using several
   independent fitting methods, as a cross-check against each other



In [8]:
import pandas as pd
import numpy as np

# Physical half-life of F-18, fixed/known constant (not estimated)
T_PHYS = 109.771  # minutes

def half_life(decay_constant):
    """Convert an exponential decay constant (1/min) to a half-life (min)."""
    return np.log(2) / decay_constant

## 1. Load and clean the raw data

The raw CSV has one row per (patient, measurement). `Patient`,
`injection dose`, `height`, `sex`, and `weight` are only populated on the
first of each patient's rows and blank on the following rows, so 
the blank rows are forward-filled, and the `BMI index` is added.

In [9]:
raw = pd.read_csv("./data/CSV.csv")

df = raw.copy()
df["Patient"] = df["Patient"].ffill()
fill_cols = ["injection dose", "height", "weight"]
if "sex" in df.columns:
    fill_cols.append("sex")
for col in fill_cols:
    df[col] = df.groupby("Patient")[col].transform(lambda s: s.ffill().bfill())

df = df.drop(columns=["injection time"])  # never collected

df["bmi"] = df["weight"] / (df["height"] / 100) ** 2

# position of each measurement within its patient (1st, 2nd, 3rd
# reading, in the order recorded)
df["pos"] = df.groupby("Patient").cumcount() + 1

print(f"Loaded {len(df)} rows, {df['Patient'].nunique()} patients")
df.head(6)

Loaded 108 rows, 36 patients


,Patient,injection dose,time,dose rate,height,weight,sex,bmi,pos
0,P01,8.9,28,30.4,161,80,Female,30.863007,1
1,P01,8.9,62,13.3,161,80,Female,30.863007,2
2,P01,8.9,106,10.0,161,80,Female,30.863007,3
3,P02,8.5,29,23.9,165,72,Female,26.446281,1
4,P02,8.5,47,18.3,165,72,Female,26.446281,2
5,P02,8.5,86,12.0,165,72,Female,26.446281,3


## 2. Resolve the a patient duplicate row

A patient has two identical rows (time=22, dose rate=29.2) and one
distinct row (time=57, dose rate=17.0) -> only 2 genuine
measurements, not 3.

Decision: drop the duplicate row and keep the patient's 2 real
measurements as-is. We do NOT fabricate a 3rd value. A
mixed-effects model (Stage 4) naturally handles patients who
contribute fewer observations than others -- no imputation needed,
and no risk of biasing the between-patient variance we're trying
to estimate honestly.

In [10]:
dup_mask = df.duplicated(subset=["Patient", "time", "dose rate"], keep="first")
print("Dropping this duplicate row:")
print(df[dup_mask])

# Identify which patient ID had the duplicate WITHOUT hardcoding it as
# "8" or "P08" -- the ID scheme in the raw file has changed at least
# once already (plain integers -> "P01".."P36"), so hardcoding a
# specific ID here is exactly the kind of thing that silently breaks
# and prints an empty result instead of erroring. Read it from the
# data itself instead.
affected_patient = df.loc[dup_mask, "Patient"].iloc[0]

df = df[~dup_mask].reset_index(drop=True)

print(f"\nPatient {affected_patient!r} after cleaning:")
print(df[df["Patient"] == affected_patient])

print("\nObservations per patient (expect all 3s, one 2 for patient 8):")
print(df.groupby("Patient").size().value_counts())

Dropping this duplicate row:
   Patient  injection dose  time  dose rate  height  weight     sex  \
22     P08             9.7    22       29.2     169      88  Female   

          bmi  pos  
22  30.811246    2  

Patient 'P08' after cleaning:
   Patient  injection dose  time  dose rate  height  weight     sex  \
21     P08             9.7    22       29.2     169      88  Female   
22     P08             9.7    57       17.0     169      88  Female   

          bmi  pos  
21  30.811246    1  
22  30.811246    3  

Observations per patient (expect all 3s, one 2 for patient 8):
3    35
2     1
Name: count, dtype: int64


## 3. Corrected population Teff/Tbio -- cross-checked across methods

Teff is estimated using three approaches: **pooled log-linear OLS**, **20-min binned averaging**, and **positional averaging**. The first two approaches estimate the population decay from dose-rate measurements, while positional averaging reproduces the original spreadsheet approach.


In [11]:
def fit_pooled_loglinear(data, dose_col="dose rate"):
    """Log-linear OLS across every individual point (Excel-trendline
    equivalent). Ignores patient clustering -- reported here only as
    a cross-check, not as the final model."""
    t = data["time"].values
    y = np.log(data[dose_col].values)
    X = np.column_stack([np.ones_like(t), t])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    lam = -beta[1]
    return lam, half_life(lam)


def fit_binned(data, dose_col="dose rate", edges=range(20, 141, 20)):
    """Bin time into 20-min windows, average dose rate across
    patients within each window, fit exponential to the bin
    averages."""
    d = data.copy()
    d["bin"] = pd.cut(d["time"], bins=list(edges), right=False)
    binned = d.groupby("bin", observed=True).agg(t=("time", "mean"), y=(dose_col, "mean"))
    t = binned["t"].values
    y = np.log(binned["y"].values)
    X = np.column_stack([np.ones_like(t), t])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    lam = -beta[1]
    return lam, half_life(lam)


def fit_positional(data, dose_col="dose rate"):
    """Average time and dose rate separately across patients within
    each of the 1st/2nd/3rd measurement slots (the method used in
    the professor's spreadsheet)."""
    pos = data.groupby("pos").agg(t=("time", "mean"), y=(dose_col, "mean"))
    t = pos["t"].values
    y = np.log(pos["y"].values)
    X = np.column_stack([np.ones_like(t), t])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    lam = -beta[1]
    return lam, half_life(lam)


results = {}
for name, fn in [
    ("Pooled log-linear OLS", fit_pooled_loglinear),
    ("Binned time-average (20-min bins)", fit_binned),
    ("Positional average (1st/2nd/3rd slot)", fit_positional),
]:
    lam, teff = fn(df)
    results[name] = teff
    print(f"{name:40s}  Teff = {teff:6.2f} min")

Pooled log-linear OLS                     Teff =  89.69 min
Binned time-average (20-min bins)         Teff =  94.08 min
Positional average (1st/2nd/3rd slot)     Teff =  82.47 min


## 4. Back out Tbio from the pooled Teff

1/Teff = 1/Tphys + 1/Tbio  ->  1/Tbio = 1/Teff - 1/Tphys

In [12]:
teff_pooled = results["Pooled log-linear OLS"]
lam_eff = np.log(2) / teff_pooled
lam_phys = np.log(2) / T_PHYS
lam_bio = lam_eff - lam_phys
tbio_pooled = np.log(2) / lam_bio

print(f"Pooled Teff = {teff_pooled:.1f} min")
print(f"Implied Tbio = {tbio_pooled:.1f} min")

Pooled Teff = 89.7 min
Implied Tbio = 490.4 min


## 5. Save the cleaned dataset for use in later stages

This is the dataset Stage 3 onward (descriptive stats, mixed
model, covariate analysis) will build on.

In [13]:
df.to_csv("./data/patient_data_clean.csv", index=False)
print("Saved patient_data_clean.csv:", df.shape)

Saved patient_data_clean.csv: (107, 9)
